In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from scipy.ndimage import label
import os
import warnings
warnings.filterwarnings('ignore')

INPUT_CSV = '/kaggle/input/datasets/risha8750/phase3-output/phase3_keypoints_master.csv'
OUTPUT_DIR = '/kaggle/working/phase4_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_SIZE = 30
STRIDE_TRAIN = 15
STRIDE_TEST  = 30
SG_WINDOW    = 7
SG_ORDER     = 2
NAN_DROP_THRESHOLD = 0.30
FPS = 30

print("Imports done. Output dir:", OUTPUT_DIR)

In [ ]:
df = pd.read_csv(INPUT_CSV)
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nSample rows:")
display(df.head(3))
print("\nSources:", df['source'].unique())
print("Actions:", df['action'].unique())
print("NaN count per column:\n", df.isnull().sum())

In [ ]:
def vec_angle(ax, ay, bx, by, cx, cy):
    """
    Vectorized angle at point B between A-B-C. Returns degrees.
    Operates on full numpy arrays — no row-by-row loop.
    """
    ba_x = ax - bx;  ba_y = ay - by
    bc_x = cx - bx;  bc_y = cy - by
    dot      = ba_x * bc_x + ba_y * bc_y
    norm_ba  = np.sqrt(ba_x**2 + ba_y**2)
    norm_bc  = np.sqrt(bc_x**2 + bc_y**2)
    denom    = norm_ba * norm_bc
    cos_a    = np.where(denom < 1e-6, np.nan, dot / denom)
    cos_a    = np.clip(cos_a, -1.0, 1.0)
    return np.degrees(np.arccos(cos_a))

print("Computing angles from raw coordinates...")

# Knee flexion L — angle at left_knee: left_hip → left_knee → left_ankle
df['knee_flexion_L'] = vec_angle(
    df['left_hip_x'].values,    df['left_hip_y'].values,
    df['left_knee_x'].values,   df['left_knee_y'].values,
    df['left_ankle_x'].values,  df['left_ankle_y'].values
)

# Knee flexion R
df['knee_flexion_R'] = vec_angle(
    df['right_hip_x'].values,   df['right_hip_y'].values,
    df['right_knee_x'].values,  df['right_knee_y'].values,
    df['right_ankle_x'].values, df['right_ankle_y'].values
)

# Hip flexion L — angle at left_hip: left_shoulder → left_hip → left_knee
df['hip_flexion_L'] = vec_angle(
    df['left_shoulder_x'].values, df['left_shoulder_y'].values,
    df['left_hip_x'].values,      df['left_hip_y'].values,
    df['left_knee_x'].values,     df['left_knee_y'].values
)

# Hip flexion R
df['hip_flexion_R'] = vec_angle(
    df['right_shoulder_x'].values, df['right_shoulder_y'].values,
    df['right_hip_x'].values,      df['right_hip_y'].values,
    df['right_knee_x'].values,     df['right_knee_y'].values
)

# Shoulder and hip midpoints — needed for trunk lean
df['shoulder_mid_x'] = (df['left_shoulder_x']  + df['right_shoulder_x']) / 2
df['shoulder_mid_y'] = (df['left_shoulder_y']  + df['right_shoulder_y']) / 2
df['hip_mid_x']      = (df['left_hip_x']        + df['right_hip_x'])      / 2
df['hip_mid_y']      = (df['left_hip_y']        + df['right_hip_y'])      / 2

# Set global variables downstream cells depend on
ANGLE_COLS = ['knee_flexion_L', 'knee_flexion_R', 'hip_flexion_L', 'hip_flexion_R']
TRUNK_LEAN_POSSIBLE = True
GROUP_KEYS = ['source', 'action', 'clip_id', 'cam_id']

print("Done. Angle columns added:", ANGLE_COLS)
print("Trunk lean: ENABLED (shoulder + hip midpoints computed)")
print("\nAngle statistics:")
print(df[ANGLE_COLS].describe().round(1))

In [ ]:
def smooth_clip(group_df):
    group_df = group_df.sort_values('frame').copy()
    n_frames = len(group_df)
    
    for col in ANGLE_COLS:
        vals = group_df[col].values.astype(float)
        
        if n_frames < SG_WINDOW:
            w = n_frames if n_frames % 2 != 0 else n_frames - 1
            w = max(w, 3)
            order = min(SG_ORDER, w - 1)
        else:
            w = SG_WINDOW
            order = SG_ORDER
        
        series = pd.Series(vals)
        series = series.interpolate(method='linear', limit_direction='both')
        
        try:
            group_df[f'{col}_smooth'] = savgol_filter(series.values, w, order)
        except Exception:
            group_df[f'{col}_smooth'] = series.values
    
    return group_df

print("smooth_clip() defined.")

In [ ]:
def compute_derivatives(group_df):
    for col in ANGLE_COLS:
        smooth_col = f'{col}_smooth'
        if smooth_col not in group_df.columns:
            continue
        
        vals = group_df[smooth_col].values.astype(float)
        
        # np.gradient needs at least 2 points — skip if clip is too short
        if len(vals) < 2:
            group_df[f'{col}_velocity'] = np.nan
            group_df[f'{col}_accel']    = np.nan
            continue
        
        vel = np.gradient(vals, 1.0 / FPS)
        group_df[f'{col}_velocity'] = vel
        
        if np.nanmax(np.abs(vel)) < 800:
            accel = np.gradient(vel, 1.0 / FPS)
            group_df[f'{col}_accel'] = accel
        else:
            group_df[f'{col}_accel'] = np.nan
    
    return group_df


def compute_asymmetry(group_df):
    if 'knee_flexion_L_smooth' in group_df.columns and 'knee_flexion_R_smooth' in group_df.columns:
        L = group_df['knee_flexion_L_smooth'].values
        R = group_df['knee_flexion_R_smooth'].values
        denom = 0.5 * (np.abs(L) + np.abs(R))
        denom = np.where(denom < 1e-6, np.nan, denom)
        group_df['asymmetry_index_smooth'] = 100 * (L - R) / denom
    return group_df


def compute_trunk_lean(group_df):
    if TRUNK_LEAN_POSSIBLE:
        dx = group_df['shoulder_mid_x'].values - group_df['hip_mid_x'].values
        dy = group_df['shoulder_mid_y'].values - group_df['hip_mid_y'].values
        group_df['trunk_lean'] = np.degrees(np.arctan2(dx, dy))
    return group_df

print("Functions redefined with length guard.")

In [ ]:
def should_drop_clip(group_df):
    n = len(group_df)
    for col in ANGLE_COLS:
        nan_frac = group_df[col].isnull().sum() / n
        if nan_frac > NAN_DROP_THRESHOLD:
            return True, col, nan_frac
    return False, None, 0.0

print("should_drop_clip() defined.")

In [ ]:
processed_clips = []
drop_log = []
clip_groups = df.groupby(GROUP_KEYS)

total = len(clip_groups)
print(f"Total clips to process: {total}")

for i, (keys, group) in enumerate(clip_groups):
    source, action, clip_id, cam_id = keys
    
    drop, bad_col, frac = should_drop_clip(group)
    if drop:
        drop_log.append({
            'source': source, 'action': action,
            'clip_id': clip_id, 'cam_id': cam_id,
            'reason': f'NaN>{NAN_DROP_THRESHOLD*100:.0f}% on {bad_col}',
            'nan_frac': round(frac, 3)
        })
        continue
    
    group = smooth_clip(group)
    group = compute_trunk_lean(group)
    group = compute_asymmetry(group)
    group = compute_derivatives(group)
    
    processed_clips.append(group)
    
    if (i + 1) % 500 == 0:
        print(f"  Processed {i+1}/{total} clips...")

print(f"\nDone. Kept: {len(processed_clips)} clips | Dropped: {len(drop_log)} clips")

df_processed = pd.concat(processed_clips, ignore_index=True)
print("Processed dataframe shape:", df_processed.shape)

In [ ]:
drop_df = pd.DataFrame(drop_log)
drop_df.to_csv(f'{OUTPUT_DIR}/phase4_drop_log.csv', index=False)
print(f"Drop log saved: {len(drop_df)} clips dropped")
if len(drop_df) > 0:
    display(drop_df.head(10))

df_processed.to_csv(f'{OUTPUT_DIR}/phase4_processed_checkpoint.csv', index=False)
print("Checkpoint saved to phase4_processed_checkpoint.csv")

In [ ]:
FEATURE_COLS = []
for col in ANGLE_COLS:
    FEATURE_COLS += [f'{col}_smooth', f'{col}_velocity', f'{col}_accel']

if TRUNK_LEAN_POSSIBLE:
    FEATURE_COLS.append('trunk_lean')

FEATURE_COLS.append('asymmetry_index_smooth')
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_processed.columns]
print(f"Feature columns for windows ({len(FEATURE_COLS)}):\n", FEATURE_COLS)

windows_train = []
windows_test  = []
window_meta_train = []
window_meta_test  = []

for keys, group in df_processed.groupby(GROUP_KEYS):
    source, action, clip_id, cam_id = keys
    group = group.sort_values('frame').reset_index(drop=True)
    feat_matrix = group[FEATURE_COLS].values.astype(float)
    n = len(feat_matrix)
    
    if n < WINDOW_SIZE:
        continue
    
    for start in range(0, n - WINDOW_SIZE + 1, STRIDE_TRAIN):
        window = feat_matrix[start:start + WINDOW_SIZE]
        nan_per_col = np.isnan(window).mean(axis=0)
        if np.any(nan_per_col > 0.20):
            continue
        windows_train.append(window)
        window_meta_train.append({
            'source': source, 'action': action,
            'clip_id': clip_id, 'cam_id': cam_id,
            'window_start_frame': start
        })
    
    for start in range(0, n - WINDOW_SIZE + 1, STRIDE_TEST):
        window = feat_matrix[start:start + WINDOW_SIZE]
        nan_per_col = np.isnan(window).mean(axis=0)
        if np.any(nan_per_col > 0.20):
            continue
        windows_test.append(window)
        window_meta_test.append({
            'source': source, 'action': action,
            'clip_id': clip_id, 'cam_id': cam_id,
            'window_start_frame': start
        })

windows_train = np.array(windows_train, dtype=np.float32)
windows_test  = np.array(windows_test,  dtype=np.float32)

print(f"\nTraining windows shape : {windows_train.shape}  → (n_windows, 30 frames, {len(FEATURE_COLS)} features)")
print(f"Test windows shape     : {windows_test.shape}")

In [ ]:
np.save(f'{OUTPUT_DIR}/phase4_windows_train.npy', windows_train)
np.save(f'{OUTPUT_DIR}/phase4_windows_test.npy',  windows_test)

meta_train_df = pd.DataFrame(window_meta_train)
meta_test_df  = pd.DataFrame(window_meta_test)
meta_train_df.to_csv(f'{OUTPUT_DIR}/phase4_meta_train.csv', index=False)
meta_test_df.to_csv(f'{OUTPUT_DIR}/phase4_meta_test.csv',   index=False)

pd.Series(FEATURE_COLS).to_csv(f'{OUTPUT_DIR}/phase4_feature_columns.csv', index=False, header=False)

print("All Phase 4 outputs saved:")
print(f"  phase4_windows_train.npy  → {windows_train.shape}")
print(f"  phase4_windows_test.npy   → {windows_test.shape}")
print(f"  phase4_meta_train.csv     → {len(meta_train_df)} rows")
print(f"  phase4_meta_test.csv      → {len(meta_test_df)} rows")
print(f"  phase4_feature_columns.csv → {len(FEATURE_COLS)} features")
print(f"  phase4_drop_log.csv       → {len(drop_df)} dropped clips")

In [ ]:
check = np.load(f'{OUTPUT_DIR}/phase4_windows_train.npy')
print("Reloaded shape:", check.shape)
print("NaN in training windows:", np.isnan(check).sum())
print("Inf in training windows:", np.isinf(check).sum())

knee_L_flat = check[:, :, 0].flatten()
print(f"\nKnee Flexion L stats:")
print(f"  Min: {np.nanmin(knee_L_flat):.1f}°  Max: {np.nanmax(knee_L_flat):.1f}°  Mean: {np.nanmean(knee_L_flat):.1f}°")

print("\nWindow count by action (train):")
print(meta_train_df.groupby('action').size().sort_values(ascending=False))

In [ ]:
print("=== df columns ===")
print(df.columns.tolist())

print("\n=== df_processed columns ===")
print(df_processed.columns.tolist())

print("\n=== ANGLE_COLS resolved to ===")
print(ANGLE_COLS)

print("\n=== FEATURE_COLS resolved to ===")
print(FEATURE_COLS)

print("\n=== Sample of df_processed (first 2 rows) ===")
display(df_processed.head(2))

In [29]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# Pick one representative clip — first hurdle or jumping clip
sample_clip = df_processed[
    df_processed['action'].isin(['hurdle', 'jumping', 'sprint'])
].groupby(['clip_id', 'cam_id']).filter(lambda x: len(x) >= 30).groupby(['clip_id', 'cam_id']).first().reset_index()

sample_keys = df_processed[
    df_processed['action'].isin(['hurdle', 'jumping', 'sprint'])
].groupby(['source','action','clip_id','cam_id']).filter(lambda x: len(x) >= 30)

first_group_keys = sample_keys.groupby(['source','action','clip_id','cam_id']).groups
first_key = list(first_group_keys.keys())[0]
sample = df_processed[
    (df_processed['source']   == first_key[0]) &
    (df_processed['action']   == first_key[1]) &
    (df_processed['clip_id']  == first_key[2]) &
    (df_processed['cam_id']   == first_key[3])
].sort_values('frame').head(60)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'Raw vs Smoothed Joint Angles\nClip: {first_key[1]} | {first_key[2]}', fontsize=14, fontweight='bold')

pairs = [
    ('knee_flexion_L', 'knee_flexion_L_smooth', 'Knee Flexion — Left'),
    ('knee_flexion_R', 'knee_flexion_R_smooth', 'Knee Flexion — Right'),
    ('hip_flexion_L',  'hip_flexion_L_smooth',  'Hip Flexion — Left'),
    ('hip_flexion_R',  'hip_flexion_R_smooth',  'Hip Flexion — Right'),
]

for ax, (raw_col, smooth_col, title) in zip(axes.flatten(), pairs):
    ax.plot(sample['frame'], sample[raw_col],    color='#aaaaaa', linewidth=1.2, label='Raw', alpha=0.8)
    ax.plot(sample['frame'], sample[smooth_col], color='#e63946', linewidth=2.0, label='Smoothed (SG)')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frame')
    ax.set_ylabel('Angle (°)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/graph1_raw_vs_smooth.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: graph1_raw_vs_smooth.png")

Saved: graph1_raw_vs_smooth.png


In [30]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Angular Velocity Distribution by Action Class', fontsize=14, fontweight='bold')

actions = df_processed['action'].unique()
colors  = plt.cm.tab10(np.linspace(0, 1, len(actions)))

for ax, col, title in zip(axes, ['knee_flexion_L_velocity', 'hip_flexion_L_velocity'],
                                  ['Left Knee Angular Velocity', 'Left Hip Angular Velocity']):
    for action, color in zip(actions, colors):
        subset = df_processed[df_processed['action'] == action][col].dropna()
        if len(subset) > 100:
            subset.clip(-600, 600).plot.kde(ax=ax, label=action, color=color, linewidth=1.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Angular Velocity (°/s)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-600, 600)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/graph2_velocity_by_action.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: graph2_velocity_by_action.png")

Saved: graph2_velocity_by_action.png


In [32]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Training Window Distribution by Action Class', fontsize=14, fontweight='bold')

action_counts = meta_train_df.groupby('action').size().sort_values(ascending=False)
source_counts = meta_train_df.groupby('source').size()

bars = axes[0].bar(action_counts.index, action_counts.values,
                   color=plt.cm.tab10(np.linspace(0, 1, len(action_counts))), edgecolor='white')
axes[0].set_title('Windows per Action', fontweight='bold')
axes[0].set_xlabel('Action')
axes[0].set_ylabel('Number of 30-Frame Windows')
axes[0].tick_params(axis='x', rotation=35)
for bar, val in zip(bars, action_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', va='bottom', fontsize=9)
axes[0].grid(True, axis='y', alpha=0.3)

axes[1].pie(source_counts.values, labels=source_counts.index,
            autopct='%1.1f%%', colors=['#457b9d', '#e63946'],
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Windows by Dataset Source', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/graph3_window_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: graph3_window_distribution.png")

Saved: graph3_window_distribution.png


In [33]:
fig, ax = plt.subplots(figsize=(12, 5))

asym_data = []
asym_labels = []

for action in sorted(df_processed['action'].unique()):
    vals = df_processed[df_processed['action'] == action]['asymmetry_index_smooth'].dropna()
    vals = vals[(vals > -100) & (vals < 100)]  # clip extreme outliers for display
    if len(vals) > 50:
        asym_data.append(vals.values)
        asym_labels.append(action)

bp = ax.boxplot(asym_data, labels=asym_labels, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))

colors = plt.cm.tab10(np.linspace(0, 1, len(asym_labels)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Perfect symmetry')
ax.axhline(15,  color='red',   linestyle=':', linewidth=1.5, alpha=0.7, label='>15% = High asymmetry threshold')
ax.axhline(-15, color='red',   linestyle=':', linewidth=1.5, alpha=0.7)

ax.set_title('Left-Right Knee Asymmetry Index by Action Class', fontsize=14, fontweight='bold')
ax.set_xlabel('Action')
ax.set_ylabel('Asymmetry Index (%)')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/graph4_asymmetry_by_action.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: graph4_asymmetry_by_action.png")

Saved: graph4_asymmetry_by_action.png
